## Qwen2.5-VL-7B BF16 LoRA — 100-step 학습 점검

이 노트북은 전체 학습 전에 데이터·증강·loss 연결을 검증하는 단계입니다. 고정 Train split에서 500개를 선택하고 각 원본에 순열 증강 1개를 추가하여 1,000개 학습 예시를 만듭니다. Validation 100개는 원본 입력 순서 그대로 유지합니다. Test 데이터는 열지 않습니다.

학습이 끝나면 저장된 단일 LoRA adapter로 고정 Validation 50개를 생성하여 공식 지표인 Exact Match Accuracy를 계산합니다.

### 1. A100과 BF16 확인

7B 모델을 양자화하지 않고 BF16 LoRA로 학습하기 위한 하드웨어 조건을 확인합니다.

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU runtime이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16을 지원하지 않는 GPU입니다.'
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {gpu_name}')
print(f'VRAM: {gpu_memory_gb:.1f} GB')
print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')

### 2. Google Drive 프로젝트 연결

현재 로컬과 같은 프로젝트 폴더 구조를 사용합니다. `PROJECT_DIR`만 실제 Drive 경로와 다르면 수정하세요.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
required_paths = [
    PROJECT_DIR / 'data' / 'train.csv',
    PROJECT_DIR / 'data' / 'train',
    PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv',
    PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv',
    PROJECT_DIR / 'src' / 'qwen_training.py',
    PROJECT_DIR / 'scripts' / 'train_qwen_lora.py',
    PROJECT_DIR / 'scripts' / 'evaluate_qwen_baseline.py',
]
for path in required_paths:
    assert path.exists(), f'필수 경로를 찾지 못했습니다: {path}'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f'Project: {PROJECT_DIR}')

### 3. 학습 패키지 설치

Colab의 CUDA PyTorch는 교체하지 않습니다. PEFT는 LoRA adapter만 학습하도록 만들고, Accelerate는 단일 A100 학습을 관리합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 qwen-vl-utils==0.0.14 peft==0.19.1
%pip uninstall -y torchao

### 4. 증강·프롬프트·정답 변환 테스트

큰 모델을 받기 전에 24개 정답과 24개 입력 순서 조합, 증강 결정성, Validation 비증강, Prompt v2 direct Answer 형식을 검사합니다.

In [ ]:
import subprocess

tests_dir = PROJECT_DIR / 'tests'
assert tests_dir.is_dir(), f'테스트 폴더를 찾지 못했습니다: {tests_dir}'
result = subprocess.run(
    [
        sys.executable, '-m', 'unittest', 'discover',
        '-s', str(tests_dir),
        '-p', 'test_qwen*.py',
        '-v',
    ],
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f'Qwen 테스트 실패 (종료 코드 {result.returncode})')

### 5. 모델을 받기 전 실제 학습 범위 확인

원본 Train 7,628개 중 seed 42로 500개를 고르고, 원본과 증강본의 이미지 순서와 Answer가 함께 바뀌었는지 한 건을 출력합니다. Validation은 증강하지 않습니다.

In [ ]:
from src.qwen_training import (
    build_training_examples,
    choose_training_samples,
    load_labeled_split_samples,
)

all_train = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv',
    expected_split='train',
)
all_validation = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv',
    expected_split='validation',
)
selected_train = choose_training_samples(all_train, limit=500, seed=42)
train_examples = build_training_examples(selected_train, augmentation_copies=1, seed=42)
validation_examples = build_training_examples(all_validation[:100], augmentation_copies=0, seed=42)
print(f'사용 가능한 Train: {len(all_train):,}')
print(f'선택한 고유 Train: {len(selected_train):,}')
print(f'증강 후 학습 예시: {len(train_examples):,}')
print(f'사용 가능한 Validation: {len(all_validation):,}')
print(f'Validation 증강 수: {len(validation_examples) - 100}')
original, augmented = train_examples[0], train_examples[1]
print('\n원본:', original.example_id, original.sample.input_names, original.sample.answer)
print('증강:', augmented.example_id, augmented.sample.input_names, augmented.sample.answer)

### 6. 100-step BF16 LoRA 학습

비전 인코더와 원본 7B 가중치는 고정하고 언어 모델 attention/MLP의 LoRA 파라미터만 학습합니다. 실제 배치 1, gradient accumulation 8이므로 optimizer 한 step은 8개 예시의 gradient를 모은 결과입니다. 25 step마다 원본 Validation 100개의 loss를 확인합니다.

`RUN_NAME` 폴더가 이미 있다면 이전 결과를 덮어쓰지 않도록 이름의 `v1`을 `v2` 등으로 바꾸세요.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'
CACHE_DIR = Path('/content/huggingface/hub')
RUN_NAME = 'smoke_seed42_500x2_100steps_v1'
RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_lora' / RUN_NAME
assert not (RUN_DIR / 'final_adapter').exists(), (
    f'기존 adapter가 있습니다. RUN_NAME을 바꾸세요: {RUN_DIR}'
)

train_command = [
    sys.executable, '-u', '-m', 'scripts.train_qwen_lora',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--train-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv'),
    '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
    '--output-dir', str(RUN_DIR),
    '--model-name', MODEL_NAME,
    '--cache-dir', str(CACHE_DIR),
    '--train-limit', '500',
    '--validation-limit', '100',
    '--augmentation-copies', '1',
    '--max-steps', '100',
    '--batch-size', '1',
    '--eval-batch-size', '1',
    '--gradient-accumulation-steps', '8',
    '--learning-rate', '2e-5',
    '--lora-r', '16',
    '--lora-alpha', '32',
    '--lora-dropout', '0.05',
    '--eval-steps', '25',
    '--save-steps', '50',
    '--seed', '42',
]
def run_streaming(command, *, cwd):
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        recent_lines.append(line.rstrip())
        recent_lines = recent_lines[-50:]
    return_code = process.wait()
    if return_code != 0:
        tail = '\n'.join(recent_lines)
        raise RuntimeError(
            f'명령 실행 실패 (종료 코드 {return_code}). 마지막 로그:\n{tail}'
        )

run_streaming(train_command, cwd=PROJECT_DIR)

### 7. 학습 결과와 loss 확인

여기서 보는 `eval_loss`는 학습 상태 진단값이며 대회 점수가 아닙니다. NaN이 없고 초기 train loss가 내려가는지를 먼저 확인합니다.

In [ ]:
import json
import pandas as pd
from IPython.display import display

run_config = json.loads((RUN_DIR / 'run_config.json').read_text(encoding='utf-8'))
training_summary = json.loads((RUN_DIR / 'training_summary.json').read_text(encoding='utf-8'))
print(json.dumps(run_config, ensure_ascii=False, indent=2))
print(json.dumps(training_summary, ensure_ascii=False, indent=2))
trainer_state_path = RUN_DIR / 'checkpoints' / 'trainer_state.json'
if trainer_state_path.exists():
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8'))
    logs = pd.DataFrame(trainer_state.get('log_history', []))
    display(logs)

### 8. 저장된 LoRA adapter의 Validation Exact Match

학습 프로세스가 끝난 뒤 base model에 저장된 단일 adapter를 다시 연결합니다. Zero-shot과 동일한 seed 42 Validation 50개를 사용하므로 직접 비교할 수 있습니다.

In [ ]:
ADAPTER_DIR = RUN_DIR / 'final_adapter'
EVAL_OUTPUT = RUN_DIR / 'generation_evaluation'
assert ADAPTER_DIR.is_dir(), f'저장된 adapter를 찾지 못했습니다: {ADAPTER_DIR}'
eval_command = [
    sys.executable, '-u', '-m', 'scripts.evaluate_qwen_baseline',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
    '--output-dir', str(EVAL_OUTPUT),
    '--model-name', MODEL_NAME,
    '--adapter-path', str(ADAPTER_DIR),
    '--prompt-version', 'v2_direct_answer',
    '--cache-dir', str(CACHE_DIR),
    '--device', 'cuda',
    '--dtype', 'bfloat16',
    '--limit', '50',
    '--seed', '42',
    '--max-new-tokens', '16',
]
run_streaming(eval_command, cwd=PROJECT_DIR)

### 9. Zero-shot과 LoRA 결과 비교

공식 지표는 `exact_match_accuracy`입니다. 학습 후 prediction collapse가 줄었는지 `unique_answer_count`와 `most_common_fraction`도 같이 확인합니다.

In [ ]:
lora_summary_path = EVAL_OUTPUT / 'validation_50' / 'summary.json'
lora_predictions_path = EVAL_OUTPUT / 'validation_50' / 'predictions.csv'
lora_summary = json.loads(lora_summary_path.read_text(encoding='utf-8'))
lora_predictions = pd.read_csv(lora_predictions_path)
comparison = [{
    'run': 'LoRA 100-step',
    'exact_match': lora_summary['metrics']['exact_match_accuracy'],
    'position_accuracy': lora_summary['metrics']['position_accuracy'],
    'unique_answers': lora_summary['predictions']['unique_answer_count'],
    'most_common_fraction': lora_summary['predictions']['most_common_fraction'],
}]
zero_path = PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_prompt_v2' / 'validation_50' / 'summary.json'
if zero_path.exists():
    zero = json.loads(zero_path.read_text(encoding='utf-8'))
    comparison.insert(0, {
        'run': '7B zero-shot',
        'exact_match': zero['metrics']['exact_match_accuracy'],
        'position_accuracy': zero['metrics']['position_accuracy'],
        'unique_answers': zero['predictions']['unique_answer_count'],
        'most_common_fraction': zero['predictions']['most_common_fraction'],
    })
display(pd.DataFrame(comparison))
display(lora_predictions.head(10))

### 다음 단계 판단 기준

- `Collator audit passed`가 출력되어야 합니다. 이는 정답 토큰만 loss에 포함됐다는 뜻입니다.
- train loss가 NaN 없이 전반적으로 내려가야 합니다.
- LoRA Exact Match가 zero-shot 10%보다 개선되는지 확인합니다.
- 100-step은 성능 확정 실험이 아닙니다. 파이프라인이 정상일 때만 전체 7,628개와 최대 3 epoch 설정으로 넘어갑니다.
- 이 노트북에서는 Test 예측이나 submission.csv를 만들지 않습니다.